# 📦 From Notebook to Standalone Executable

**Itanna — Electrical Engineering for VSCode**

This notebook demonstrates how to create a standalone executable from a Python function using the Itanna executable builder.

**The flow:**
1. Define a Python function (with Tkinter GUI)
2. Call `build_app()` to compile it with Nuitka
3. Deploy the standalone binary

## Step 1: Define Your Application

In [ ]:
from electrical.executable.builder import tkinter_template, build_app
from electrical.converter.buck import design_buck

# Create the GUI body (tkinter code)
ui_body = '''
frame = ttk.Frame(root, padding="20")
frame.pack(fill="both", expand=True)

ttk.Label(frame, text="Buck Converter Calculator",
          font=("Helvetica", 16)).pack(pady=(0, 20))

# Input fields
inputs = [
    ("Input Voltage (V):", "vin", "12"),
    ("Output Voltage (V):", "vout", "5"),
    ("Output Current (A):", "iout", "2"),
    ("Frequency (kHz):", "fsw", "300"),
    ("Ripple (mV):", "vripple", "10"),
]

entries = {}
for label, key, default in inputs:
    row = ttk.Frame(frame)
    row.pack(fill="x", pady=2)
    ttk.Label(row, text=label, width=20).pack(side="left")
    entries[key] = ttk.Entry(row)
    entries[key].insert(0, default)
    entries[key].pack(side="right", fill="x", expand=True)

result_text = tk.Text(frame, height=10, width=60)
result_text.pack(pady=10)

def on_calculate():
    try:
        v = float(entries["vin"].get())
        vo = float(entries["vout"].get())
        i = float(entries["iout"].get())
        f = float(entries["fsw"].get()) * 1e3
        vr = float(entries["vripple"].get()) * 1e-3
        r = design_buck(v, vo, i, f, vr)
        result_text.delete("1.0", "end")
        for k, val in r.items():
            if isinstance(val, float):
                result_text.insert("end", f"{k:25s}: {val:.4e}\n")
            else:
                result_text.insert("end", f"{k:25s}: {val}\n")
    except Exception as ex:
        result_text.delete("1.0", "end")
        result_text.insert("end", f"Error: {ex}")

ttk.Button(frame, text="Calculate", command=on_calculate).pack(pady=10)
ttk.Button(frame, text="Quit", command=root.destroy).pack()
'''

# Generate the source code
source = tkinter_template(
    title="Buck Converter Calculator",
    width=500,
    height=500,
    body=ui_body,
)

print(f"Source code generated! ({len(source)} bytes)")
print(source[:200] + "...")

## Step 2: Build Standalone Executable

In [ ]:
# NOTE: Nuitka must be installed for this to work
# Install with: pip install nuitka

from electrical.executable.builder import build_app

# Build from template
result = build_app(
    func=None,  # Using template
    name="buck-calculator",
    gui="tkinter",
    output_dir="~/Desktop/",
    template_args={
        "title": "Buck Converter Calculator",
        "width": 500,
        "height": 500,
        "body": ui_body,
    },
)

if result["status"] == "success":
    print(f"✅ Executable built: {result['binary_path']}")
elif result["status"] == "error":
    print(f"❌ Build failed: {result.get('error', 'unknown error')}")
else:
    print(f"Result: {result}")

## Appendix: Function-Based API (Simpler)

If you already have a function, use this shorter approach:

In [ ]:
def my_simple_app():
    import tkinter as tk
    from tkinter import ttk
    root = tk.Tk()
    root.title("Simple App")
    ttk.Label(root, text="Hello from Itanna!").pack()
    root.mainloop()

# Uncomment to build:
# result = build_app(my_simple_app, name="simple-app", gui="tkinter", output_dir="~/Desktop/")
# print(result)

---
**Commands:**  
- `Ctrl+Alt+N` — New EE Notebook  
- `Ctrl+Alt+B` — Insert Buck Calculator snippet  
- `Ctrl+Alt+X` — Insert Executable Builder snippet  
- `Ctrl+Alt+W` — Welcome Page